# TCGA KIDNEY DATA RETRIEVAL AND NSGA-II PIPELINE

NOTES:

This notebook uses the GDC STAR-Counts files because each tab-delimited file contains raw unstranded counts, tpm_unstranded, gene identifiers, and gene annotations. It queries file metadata through the GDC /files endpoint and downloads each open-access file by its UUID through the /data endpoint.

Data is split 70/30 only once.  Then reuses that identical 70/30 split for both expression branches. Each branch performs its own training-only filtering and MAD ranking.

log2(TPM + 1) as the primary pipeline, with code to download and preserve raw counts and then add a parallel log2(CPM + 1) sensitivity analysis.


#What the code performs

The notebook includes all steps from download through ready-to-model training and validation files:

1. Queries TCGA-KICH, TCGA-KIRC, and TCGA-KIRP.
2. Restricts the query to:
   * Primary Tumor samples
   * RNA-Seq
   * Gene Expression Quantification
   * STAR-Counts
   * Open-access TSV files
3. Retains one tumor RNA-seq file per patient.
4. Downloads files with:
   * Automatic retries
   * Resume behavior
   * MD5 verification
5. Extracts both:
   * Raw unstranded counts
   * GDC tpm_unstranded
6. Retains protein-coding genes.
7. Calculates each sample’s CPM denominator from all annotated Ensembl genes before restricting the predictor matrix to protein-coding genes.
8. Creates one stratified 70/30 patient split.
9. Reuses that exact split for both branches.
10. Runs the primary branch:

    log_2 (TPM+1)

11. Runs the sensitivity branch:

    log_2 (CPM+1)

12. In each branch independently, using training data only:
   * Retains genes with expression greater than 1 in at least 10% of training patients.
   * Calculates MAD.
   * Retains the top 3,000 genes by MAD.
13. Applies each training-derived gene list to the corresponding held-out validation data.
14. Saves gene-selection audit tables and branch-comparison summaries.
15. Includes optional training-only standardization code for SVM, logistic regression, and other scale-sensitive classifiers.

#Main ready-to-model outputs

The notebook produces:

    primary_log2_tpm_training.csv
    primary_log2_tpm_heldout_validation.csv
    primary_log2_tpm_selected_genes.csv

    sensitivity_log2_cpm_training.csv
    sensitivity_log2_cpm_heldout_validation.csv
    sensitivity_log2_cpm_selected_genes.csv

The full source matrices are compressed because they may contain approximately 20,000 protein-coding gene columns. The final 3,000-gene training and validation matrices are saved as ordinary uncompressed CSV files for direct use in the NSGA-II notebook.

The main settings are collected in Cell 1. The defaults are:

    RANDOM_SEED = 42
    VALIDATION_FRACTION = 0.30
    EXPRESSION_THRESHOLD = 1.0
    MIN_TRAINING_PROPORTION_ABOVE_THRESHOLD = 0.10
    N_TOP_MAD_GENES = 3000

In [1]:
"""TCGA kidney cancer RNA-seq TPM/CPM preprocessing workflow."""

# # TCGA Kidney Cancer RNA-seq Download and Preprocessing
#
# This Google Colab notebook downloads open-access **GDC STAR-Counts** files for:
#
# - TCGA-KICH
# - TCGA-KIRC
# - TCGA-KIRP
#
# It creates two parallel, leakage-controlled preprocessing branches:
#
# 1. **Primary analysis:** `log2(TPM + 1)`
# 2. **Sensitivity analysis:** `log2(CPM + 1)` calculated from preserved raw unstranded counts
#
# Both branches use the **same patient-level stratified 70/30 training-validation split**.
# Low-expression filtering and median absolute deviation (MAD) ranking are learned from the training set only.

'TCGA kidney cancer RNA-seq TPM/CPM preprocessing workflow.'

# CELL 1: IMPORTS, REPRODUCIBILITY, AND USER SETTINGS

In [2]:

# ============================================================
# CELL 1: IMPORTS, REPRODUCIBILITY, AND USER SETTINGS
# ============================================================

# This notebook uses packages that are normally preinstalled in Google Colab.
# Do not upgrade scikit-learn unless Colab specifically reports that it is missing,
# because mixing incompatible package versions can produce import errors.

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import hashlib
import json
import os
import platform
import random
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [3]:
# ----------------------------
# Reproducibility settings
# ----------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ----------------------------
# TCGA projects and class labels
# ----------------------------
PROJECT_TO_LABEL = {
    "TCGA-KICH": "KICH",
    "TCGA-KIRC": "KIRC",
    "TCGA-KIRP": "KIRP",
}

# ----------------------------
# Validation design
# ----------------------------
VALIDATION_FRACTION = 0.30

# ----------------------------
# Training-only expression filter
# ----------------------------
# Retain a gene when expression is greater than 1 in at least 10% of
# TRAINING patients. This is equivalent to removing genes having
# expression <= 1 in at least 90% of training patients.
EXPRESSION_THRESHOLD = 1.0
MIN_TRAINING_PROPORTION_ABOVE_THRESHOLD = 0.10

# ----------------------------
# Training-only variability ranking
# ----------------------------
# Choose a value in the previously recommended range of approximately
# 2,000 to 5,000 genes. Change this one setting if desired.
N_TOP_MAD_GENES = 3000

In [4]:
# ----------------------------
# Download settings
# ----------------------------
MAX_DOWNLOAD_WORKERS = 6
MAX_DOWNLOAD_ATTEMPTS = 5
DOWNLOAD_CHUNK_SIZE_BYTES = 1024 * 1024  # 1 MB

# ----------------------------
# Output settings
# ----------------------------
# The complete source matrices are wide and can be large.
# Compressed CSV preserves a standard CSV representation while reducing storage.
SAVE_SOURCE_MATRICES_AS_COMPRESSED_CSV = True

# Set this to True only if an uncompressed full matrix is specifically required.
# The selected 3,000-gene training and validation files are always saved as
# ordinary, uncompressed CSV files later in the notebook.
SAVE_SOURCE_MATRICES_AS_UNCOMPRESSED_CSV = False

# Parquet is optional. It is usually faster to reload, but extremely wide
# matrices can be less portable than CSV.
SAVE_SOURCE_MATRICES_AS_PARQUET = False

# Saving the complete unfiltered log matrices is optional because the final
# selected training/validation matrices are saved automatically.
SAVE_FULL_TRANSFORMED_MATRICES = False

In [5]:
# ----------------------------
# Select the working directory
# ----------------------------
# By default, files are written to temporary Colab storage under /content.
# To retain files in Google Drive, change USE_GOOGLE_DRIVE to True.
USE_GOOGLE_DRIVE = True

try:
    from google.colab import drive  # type: ignore
    IN_GOOGLE_COLAB = True
except ImportError:
    IN_GOOGLE_COLAB = False

if USE_GOOGLE_DRIVE:
    if not IN_GOOGLE_COLAB:
        raise RuntimeError(
            "USE_GOOGLE_DRIVE=True, but this notebook is not running in Google Colab."
        )
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/TCGA_Kidney_RNASeq")
elif IN_GOOGLE_COLAB:
    BASE_DIR = Path("/content/TCGA_Kidney_RNASeq")
else:
    BASE_DIR = Path.cwd() / "TCGA_Kidney_RNASeq"

DOWNLOAD_DIR = BASE_DIR / "gdc_star_counts_files"
OUTPUT_DIR = BASE_DIR / "processed_outputs"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Working directory: {BASE_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Mounted at /content/drive
Python version: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
pandas version: 2.2.2
NumPy version: 2.0.2
Working directory: /content/drive/MyDrive/TCGA_Kidney_RNASeq
Output directory: /content/drive/MyDrive/TCGA_Kidney_RNASeq/processed_outputs


# CELL 2: GDC API QUERY HELPERS

In [6]:

# ============================================================
# CELL 2: GDC API QUERY HELPERS
# ============================================================

GDC_FILES_ENDPOINT = "https://api.gdc.cancer.gov/files"
GDC_DATA_ENDPOINT = "https://api.gdc.cancer.gov/data"


def make_in_filter(field, values):
    """
    Construct one GDC API 'in' filter.

    Parameters
    ----------
    field : str
        GDC field name.
    values : list
        Accepted values for the field.

    Returns
    -------
    dict
        Filter dictionary accepted by the GDC API.
    """
    return {
        "op": "in",
        "content": {
            "field": field,
            "value": list(values),
        },
    }


def create_requests_session():
    """
    Create a requests Session with automatic retry behavior.

    This is used for metadata queries. File downloads also have their own
    explicit retry loop because a partially downloaded file must be removed
    before another attempt.
    """
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET", "POST"}),
        raise_on_status=False,
    )

    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"User-Agent": "TCGA-kidney-RNASeq-Colab-workflow/1.0"})
    return session


def query_star_counts_files(project_id):
    """
    Query open-access primary-tumor STAR-Counts files for one TCGA project.

    The query requests:
      - Transcriptome Profiling
      - Gene Expression Quantification
      - RNA-Seq
      - STAR - Counts workflow
      - TSV format
      - Open access
      - Primary Tumor samples

    Parameters
    ----------
    project_id : str
        TCGA project ID, such as 'TCGA-KIRC'.

    Returns
    -------
    list of dict
        Raw GDC file records.
    """
    filters = {
        "op": "and",
        "content": [
            make_in_filter("cases.project.project_id", [project_id]),
            make_in_filter("files.data_category", ["Transcriptome Profiling"]),
            make_in_filter("files.data_type", ["Gene Expression Quantification"]),
            make_in_filter("files.experimental_strategy", ["RNA-Seq"]),
            make_in_filter("files.analysis.workflow_type", ["STAR - Counts"]),
            make_in_filter("files.data_format", ["TSV"]),
            make_in_filter("files.access", ["open"]),
            make_in_filter("cases.samples.sample_type", ["Primary Tumor"]),
        ],
    }

    fields = [
        "file_id",
        "file_name",
        "file_size",
        "md5sum",
        "access",
        "data_category",
        "data_type",
        "data_format",
        "experimental_strategy",
        "analysis.workflow_type",
        "cases.case_id",
        "cases.submitter_id",
        "cases.project.project_id",
        "cases.samples.sample_id",
        "cases.samples.submitter_id",
        "cases.samples.sample_type",
        "cases.samples.tissue_type",
    ]

    payload = {
        "filters": filters,
        "format": "JSON",
        "fields": ",".join(fields),
        # Each kidney project contains far fewer than 10,000 matching files.
        "size": 10000,
    }

    session = create_requests_session()
    response = session.post(
        GDC_FILES_ENDPOINT,
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=(30, 180),
    )
    response.raise_for_status()

    result = response.json()
    hits = result.get("data", {}).get("hits", [])

    if not hits:
        raise RuntimeError(
            f"The GDC query returned no matching files for {project_id}. "
            "Check the API response and current GDC field values."
        )

    return hits


def flatten_gdc_file_hit(hit, queried_project_id):
    """
    Convert one nested GDC file record into one or more flat records.

    A file is normally associated with one case. The function is written to
    handle multiple cases defensively. Within each case, it selects a Primary
    Tumor sample for descriptive metadata.

    Parameters
    ----------
    hit : dict
        One file record returned by the GDC API.
    queried_project_id : str
        Project used in the query.

    Returns
    -------
    list of dict
        Flat metadata records.
    """
    flat_records = []
    cases = hit.get("cases", []) or []

    for case in cases:
        samples = case.get("samples", []) or []
        primary_samples = [
            sample for sample in samples
            if sample.get("sample_type") == "Primary Tumor"
        ]

        # The API query already requires a primary tumor relationship.
        # This additional check prevents accidental use of a non-primary sample.
        if not primary_samples:
            continue

        # If more than one primary tumor sample is attached to the returned
        # relationship, choose deterministically by the submitted TCGA barcode.
        primary_samples = sorted(
            primary_samples,
            key=lambda sample: (
                str(sample.get("submitter_id", "")),
                str(sample.get("sample_id", "")),
            ),
        )
        sample = primary_samples[0]

        project_id = (
            case.get("project", {}).get("project_id")
            or queried_project_id
        )

        flat_records.append(
            {
                "project_id": project_id,
                "class_label": PROJECT_TO_LABEL[project_id],
                "case_id": case.get("case_id"),
                "case_submitter_id": case.get("submitter_id"),
                "sample_id": sample.get("sample_id"),
                "sample_submitter_id": sample.get("submitter_id"),
                "sample_type": sample.get("sample_type"),
                "tissue_type": sample.get("tissue_type"),
                "file_id": hit.get("file_id") or hit.get("id"),
                "file_name": hit.get("file_name"),
                "file_size": hit.get("file_size"),
                "md5sum": hit.get("md5sum"),
                "access": hit.get("access"),
                "workflow_type": hit.get("analysis", {}).get("workflow_type"),
            }
        )

    return flat_records

# CELL 3: QUERY ALL THREE PROJECTS AND SELECT ONE FILE PER PATIENT

In [7]:

# ============================================================
# CELL 3: QUERY ALL THREE PROJECTS AND SELECT ONE FILE PER PATIENT
# ============================================================

all_flat_records = []

for project_id in PROJECT_TO_LABEL:
    print(f"Querying {project_id}...")
    project_hits = query_star_counts_files(project_id)

    project_records = []
    for hit in project_hits:
        project_records.extend(flatten_gdc_file_hit(hit, project_id))

    print(
        f"  GDC file hits: {len(project_hits):,}; "
        f"usable primary-tumor relationships: {len(project_records):,}"
    )
    all_flat_records.extend(project_records)

gdc_metadata_all = pd.DataFrame(all_flat_records)

required_metadata_columns = [
    "project_id",
    "class_label",
    "case_submitter_id",
    "sample_submitter_id",
    "sample_type",
    "file_id",
    "file_name",
    "md5sum",
]

missing_metadata_columns = [
    column for column in required_metadata_columns
    if column not in gdc_metadata_all.columns
]
if missing_metadata_columns:
    raise RuntimeError(
        f"Required metadata columns are missing: {missing_metadata_columns}"
    )

# Remove records that lack identifiers needed for patient-level processing.
gdc_metadata_all = gdc_metadata_all.dropna(
    subset=[
        "project_id",
        "class_label",
        "case_submitter_id",
        "file_id",
        "file_name",
    ]
).copy()

# Remove exact repeated file-patient relationships, if any.
gdc_metadata_all = gdc_metadata_all.drop_duplicates(
    subset=["case_submitter_id", "file_id"]
).copy()

# Confirm that only the intended sample and workflow types remain.
unexpected_sample_types = set(gdc_metadata_all["sample_type"].dropna()) - {"Primary Tumor"}
if unexpected_sample_types:
    raise RuntimeError(
        f"Unexpected sample types were returned: {sorted(unexpected_sample_types)}"
    )

unexpected_workflows = set(gdc_metadata_all["workflow_type"].dropna()) - {"STAR - Counts"}
if unexpected_workflows:
    raise RuntimeError(
        f"Unexpected workflow types were returned: {sorted(unexpected_workflows)}"
    )

# ------------------------------------------------------------
# Retain one primary-tumor RNA-seq file per patient.
#
# A patient can occasionally have multiple aliquots or files. The following
# deterministic sorting rule keeps the lexicographically first sample barcode,
# file name, and file UUID. No outcome information is used in this choice.
# ------------------------------------------------------------
gdc_metadata_all = gdc_metadata_all.sort_values(
    by=[
        "project_id",
        "case_submitter_id",
        "sample_submitter_id",
        "file_name",
        "file_id",
    ],
    kind="stable",
).reset_index(drop=True)

files_per_patient = gdc_metadata_all.groupby("case_submitter_id").size()
patients_with_multiple_files = files_per_patient[files_per_patient > 1]

gdc_metadata_selected = (
    gdc_metadata_all
    .drop_duplicates(subset=["case_submitter_id"], keep="first")
    .reset_index(drop=True)
)

# Verify that patient IDs are unique after selection.
if gdc_metadata_selected["case_submitter_id"].duplicated().any():
    raise RuntimeError("Patient IDs are not unique after one-file-per-patient selection.")

# Verify that all three classes are represented.
observed_projects = set(gdc_metadata_selected["project_id"])
expected_projects = set(PROJECT_TO_LABEL)
if observed_projects != expected_projects:
    raise RuntimeError(
        f"Expected projects {sorted(expected_projects)}, "
        f"but observed {sorted(observed_projects)}."
    )

print("\nOne-file-per-patient summary:")
print(
    gdc_metadata_selected.groupby(["project_id", "class_label"])
    .size()
    .rename("patients")
    .to_frame()
)

print(
    f"\nPatients with more than one eligible file before selection: "
    f"{len(patients_with_multiple_files):,}"
)
print(f"Final number of unique patients: {len(gdc_metadata_selected):,}")

# Save both the complete eligible metadata and the final selected manifest.
gdc_metadata_all.to_csv(
    OUTPUT_DIR / "gdc_star_counts_all_eligible_files.csv",
    index=False,
)
gdc_metadata_selected.to_csv(
    OUTPUT_DIR / "gdc_star_counts_one_file_per_patient.csv",
    index=False,
)

display(gdc_metadata_selected.head())


Querying TCGA-KICH...
  GDC file hits: 66; usable primary-tumor relationships: 66
Querying TCGA-KIRC...
  GDC file hits: 541; usable primary-tumor relationships: 541
Querying TCGA-KIRP...
  GDC file hits: 290; usable primary-tumor relationships: 290

One-file-per-patient summary:
                        patients
project_id class_label          
TCGA-KICH  KICH               66
TCGA-KIRC  KIRC              533
TCGA-KIRP  KIRP              290

Patients with more than one eligible file before selection: 4
Final number of unique patients: 889


,project_id,class_label,case_id,case_submitter_id,sample_id,sample_submitter_id,sample_type,tissue_type,file_id,file_name,file_size,md5sum,access,workflow_type
0,TCGA-KICH,KICH,83b3060c-2449-4581-88ef-817d126e4525,TCGA-KL-8323,8aa1a5f6-3548-494a-857b-e08124aeb9a9,TCGA-KL-8323-01A,Primary Tumor,Tumor,a4b800e3-b6ff-4823-86da-ea9755d88eac,a3bd8e23-8f88-4d92-871e-57878d42048e.rna_seq.a...,4232251,ab05a17399b6167934c7428abdd87e65,open,STAR - Counts
1,TCGA-KICH,KICH,0697436f-e487-45db-b0bc-ad9246f70196,TCGA-KL-8324,16c2976b-c2c5-40f0-8d7d-8ecfb4a15317,TCGA-KL-8324-01A,Primary Tumor,Tumor,7b5bdde2-cbc5-4f11-9dff-3217eb69734c,6c0c2bda-2fd3-4294-8e4a-7ffad2c8f16e.rna_seq.a...,4225880,d7d528d88c38b2476d0d62c5824e3389,open,STAR - Counts
2,TCGA-KICH,KICH,6101ffe6-2ef6-4256-9d6b-a7c545836995,TCGA-KL-8325,79c48bea-33e0-49e6-84f7-6103da654642,TCGA-KL-8325-01A,Primary Tumor,Tumor,f65e0c6a-05c4-409b-a446-85b90671c06e,9ec7c4f6-5fdd-4307-a5a9-b93a18e03ee0.rna_seq.a...,4231750,6887ae3affba94de5ef870c8ff808ae4,open,STAR - Counts
3,TCGA-KICH,KICH,03f3dd32-e1d1-485b-a968-3f55798f4d46,TCGA-KL-8326,59703768-8be1-47a7-8c7d-d794c0936019,TCGA-KL-8326-01A,Primary Tumor,Tumor,9452f420-4244-4518-b607-3442361d065a,3dc24ff9-20bd-4c63-8152-7e43a0804943.rna_seq.a...,4216775,810b40c7ab72f8f5f313443398773e52,open,STAR - Counts
4,TCGA-KICH,KICH,02979422-5149-4750-ad5f-483e0bec6ac5,TCGA-KL-8327,f35e66ec-7aa5-477c-abf7-07c35c90f7de,TCGA-KL-8327-01A,Primary Tumor,Tumor,0161b0d1-3fb0-45e8-bed0-df99aacaaac1,718f1665-6b2c-4d07-9dc5-93ca8e5c2bb0.rna_seq.a...,4205351,6529d9d90e70e534246f1871c69d1e09,open,STAR - Counts


# CELL 4: DOWNLOAD THE SELECTED OPEN-ACCESS STAR-COUNTS FILES

In [8]:
# ============================================================
# CELL 4: DOWNLOAD THE SELECTED OPEN-ACCESS STAR-COUNTS FILES
# ============================================================

def calculate_md5(file_path, chunk_size=1024 * 1024):
    """
    Calculate the MD5 checksum of a local file without loading it all into memory.
    """
    md5 = hashlib.md5()
    with open(file_path, "rb") as input_file:
        while True:
            chunk = input_file.read(chunk_size)
            if not chunk:
                break
            md5.update(chunk)
    return md5.hexdigest()


def sanitize_filename(file_name):
    """
    Remove path separators and unusual characters from a downloaded file name.
    """
    file_name = Path(str(file_name)).name
    return re.sub(r"[^A-Za-z0-9._-]+", "_", file_name)


def download_one_gdc_file(record):
    """
    Download one open-access GDC file with retry and MD5 verification.

    Parameters
    ----------
    record : dict
        Flat metadata record containing file_id, file_name, project_id, and md5sum.

    Returns
    -------
    dict
        Download status and final local path.
    """
    file_id = str(record["file_id"])
    project_id = str(record["project_id"])
    expected_md5 = str(record.get("md5sum") or "").lower()
    safe_name = sanitize_filename(record["file_name"])

    project_directory = DOWNLOAD_DIR / project_id
    project_directory.mkdir(parents=True, exist_ok=True)

    # Prefixing with the file UUID prevents accidental filename collisions.
    destination = project_directory / f"{file_id}__{safe_name}"
    temporary_destination = destination.with_suffix(destination.suffix + ".part")

    # Skip a correctly downloaded file so the notebook can be resumed.
    if destination.exists():
        if expected_md5:
            observed_md5 = calculate_md5(destination)
            if observed_md5 == expected_md5:
                return {
                    "file_id": file_id,
                    "status": "already_present",
                    "local_path": str(destination),
                }
        else:
            return {
                "file_id": file_id,
                "status": "already_present_no_md5",
                "local_path": str(destination),
            }

    url = f"{GDC_DATA_ENDPOINT}/{file_id}"

    for attempt in range(1, MAX_DOWNLOAD_ATTEMPTS + 1):
        try:
            if temporary_destination.exists():
                temporary_destination.unlink()

            with requests.get(
                url,
                stream=True,
                timeout=(30, 600),
                headers={"User-Agent": "TCGA-kidney-RNASeq-Colab-workflow/1.0"},
            ) as response:
                response.raise_for_status()

                with open(temporary_destination, "wb") as output_file:
                    for chunk in response.iter_content(
                        chunk_size=DOWNLOAD_CHUNK_SIZE_BYTES
                    ):
                        if chunk:
                            output_file.write(chunk)

            # Verify the completed temporary file before renaming it.
            if expected_md5:
                observed_md5 = calculate_md5(temporary_destination)
                if observed_md5 != expected_md5:
                    raise IOError(
                        f"MD5 mismatch for {file_id}: "
                        f"expected {expected_md5}, observed {observed_md5}"
                    )

            temporary_destination.replace(destination)

            return {
                "file_id": file_id,
                "status": "downloaded",
                "local_path": str(destination),
            }

        except Exception as error:
            if temporary_destination.exists():
                temporary_destination.unlink()

            if attempt == MAX_DOWNLOAD_ATTEMPTS:
                return {
                    "file_id": file_id,
                    "status": "failed",
                    "local_path": None,
                    "error": repr(error),
                }

            # Exponential backoff: 2, 4, 8, 16 seconds...
            time.sleep(2 ** attempt)

    raise RuntimeError("Unexpected download-loop termination.")


download_records = gdc_metadata_selected.to_dict(orient="records")
download_results = []

print(
    f"Downloading or verifying {len(download_records):,} files "
    f"with {MAX_DOWNLOAD_WORKERS} workers..."
)

with ThreadPoolExecutor(max_workers=MAX_DOWNLOAD_WORKERS) as executor:
    future_to_file_id = {
        executor.submit(download_one_gdc_file, record): record["file_id"]
        for record in download_records
    }

    for future in tqdm(
        as_completed(future_to_file_id),
        total=len(future_to_file_id),
        desc="GDC downloads",
    ):
        download_results.append(future.result())

download_results = pd.DataFrame(download_results)

failed_downloads = download_results.loc[
    download_results["status"] == "failed"
].copy()

print("\nDownload status:")
print(download_results["status"].value_counts(dropna=False))

if not failed_downloads.empty:
    display(failed_downloads)
    raise RuntimeError(
        f"{len(failed_downloads)} file(s) failed to download. "
        "Rerun this cell to retry them."
    )

# Attach the verified local path to each selected patient/file record.
gdc_metadata_selected = gdc_metadata_selected.merge(
    download_results[["file_id", "local_path"]],
    on="file_id",
    how="left",
    validate="one_to_one",
)

if gdc_metadata_selected["local_path"].isna().any():
    raise RuntimeError("At least one selected file does not have a local path.")

if not gdc_metadata_selected["local_path"].map(lambda path: Path(path).exists()).all():
    raise RuntimeError("At least one downloaded file is missing from local storage.")

gdc_metadata_selected.to_csv(
    OUTPUT_DIR / "gdc_star_counts_downloaded_one_file_per_patient.csv",
    index=False,
)

display(gdc_metadata_selected.head())


GDC downloads:   0%|          | 0/889 [00:00<?, ?it/s]


Download status:
status
downloaded    889
Name: count, dtype: int64


,project_id,class_label,case_id,case_submitter_id,sample_id,sample_submitter_id,sample_type,tissue_type,file_id,file_name,file_size,md5sum,access,workflow_type,local_path
0,TCGA-KICH,KICH,83b3060c-2449-4581-88ef-817d126e4525,TCGA-KL-8323,8aa1a5f6-3548-494a-857b-e08124aeb9a9,TCGA-KL-8323-01A,Primary Tumor,Tumor,a4b800e3-b6ff-4823-86da-ea9755d88eac,a3bd8e23-8f88-4d92-871e-57878d42048e.rna_seq.a...,4232251,ab05a17399b6167934c7428abdd87e65,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...
1,TCGA-KICH,KICH,0697436f-e487-45db-b0bc-ad9246f70196,TCGA-KL-8324,16c2976b-c2c5-40f0-8d7d-8ecfb4a15317,TCGA-KL-8324-01A,Primary Tumor,Tumor,7b5bdde2-cbc5-4f11-9dff-3217eb69734c,6c0c2bda-2fd3-4294-8e4a-7ffad2c8f16e.rna_seq.a...,4225880,d7d528d88c38b2476d0d62c5824e3389,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...
2,TCGA-KICH,KICH,6101ffe6-2ef6-4256-9d6b-a7c545836995,TCGA-KL-8325,79c48bea-33e0-49e6-84f7-6103da654642,TCGA-KL-8325-01A,Primary Tumor,Tumor,f65e0c6a-05c4-409b-a446-85b90671c06e,9ec7c4f6-5fdd-4307-a5a9-b93a18e03ee0.rna_seq.a...,4231750,6887ae3affba94de5ef870c8ff808ae4,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...
3,TCGA-KICH,KICH,03f3dd32-e1d1-485b-a968-3f55798f4d46,TCGA-KL-8326,59703768-8be1-47a7-8c7d-d794c0936019,TCGA-KL-8326-01A,Primary Tumor,Tumor,9452f420-4244-4518-b607-3442361d065a,3dc24ff9-20bd-4c63-8152-7e43a0804943.rna_seq.a...,4216775,810b40c7ab72f8f5f313443398773e52,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...
4,TCGA-KICH,KICH,02979422-5149-4750-ad5f-483e0bec6ac5,TCGA-KL-8327,f35e66ec-7aa5-477c-abf7-07c35c90f7de,TCGA-KL-8327-01A,Primary Tumor,Tumor,0161b0d1-3fb0-45e8-bed0-df99aacaaac1,718f1665-6b2c-4d07-9dc5-93ca8e5c2bb0.rna_seq.a...,4205351,6529d9d90e70e534246f1871c69d1e09,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...


# CELL 5: PARSE STAR-COUNTS FILES AND BUILD MATCHED MATRICES

In [9]:
# ============================================================
# CELL 5: PARSE STAR-COUNTS FILES AND BUILD MATCHED MATRICES
# ============================================================

REQUIRED_STAR_COLUMNS = {
    "gene_id",
    "gene_name",
    "gene_type",
    "unstranded",
    "tpm_unstranded",
}


def read_star_counts_file(file_path):
    """
    Read one GDC STAR-Counts TSV file.

    Returns
    -------
    protein_coding : pandas.DataFrame
        Protein-coding genes with raw unstranded counts and GDC TPM.
    library_size_all_annotated_genes : float
        Sum of raw unstranded counts across all rows with Ensembl gene IDs.

    Notes
    -----
    STAR summary rows such as N_unmapped are excluded because their IDs do not
    begin with 'ENSG'. The CPM denominator is computed from all annotated genes
    BEFORE restricting the feature matrix to protein-coding genes.
    """
    file_path = Path(file_path)

    star_data = pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        low_memory=False,
    )

    missing_columns = REQUIRED_STAR_COLUMNS - set(star_data.columns)
    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: {sorted(missing_columns)}"
        )

    # Keep only ordinary Ensembl gene rows, excluding STAR summary rows.
    gene_rows = star_data.loc[
        star_data["gene_id"].astype(str).str.startswith("ENSG")
    ].copy()

    gene_rows["unstranded"] = pd.to_numeric(
        gene_rows["unstranded"],
        errors="raise",
    )
    gene_rows["tpm_unstranded"] = pd.to_numeric(
        gene_rows["tpm_unstranded"],
        errors="raise",
    )

    library_size_all_annotated_genes = float(gene_rows["unstranded"].sum())

    if not np.isfinite(library_size_all_annotated_genes):
        raise ValueError(f"Non-finite library size in {file_path.name}.")
    if library_size_all_annotated_genes <= 0:
        raise ValueError(f"Non-positive library size in {file_path.name}.")

    protein_coding = gene_rows.loc[
        gene_rows["gene_type"].eq("protein_coding"),
        [
            "gene_id",
            "gene_name",
            "gene_type",
            "unstranded",
            "tpm_unstranded",
        ],
    ].copy()

    if protein_coding.empty:
        raise ValueError(f"No protein-coding genes found in {file_path.name}.")

    # Duplicate Ensembl gene IDs are not expected. If encountered, combine them
    # deterministically so every matrix has one column per Ensembl gene ID.
    if protein_coding["gene_id"].duplicated().any():
        protein_coding = (
            protein_coding.groupby("gene_id", as_index=False)
            .agg(
                gene_name=("gene_name", "first"),
                gene_type=("gene_type", "first"),
                unstranded=("unstranded", "sum"),
                tpm_unstranded=("tpm_unstranded", "sum"),
            )
        )

    protein_coding = protein_coding.sort_values(
        "gene_id",
        kind="stable",
    ).reset_index(drop=True)

    return protein_coding, library_size_all_annotated_genes


# Use the deterministic metadata order as the matrix row order.
gdc_metadata_selected = gdc_metadata_selected.sort_values(
    ["project_id", "case_submitter_id"],
    kind="stable",
).reset_index(drop=True)

number_of_samples = len(gdc_metadata_selected)

# Read the first file to establish a reference gene order.
first_file_path = gdc_metadata_selected.loc[0, "local_path"]
first_gene_data, first_library_size = read_star_counts_file(first_file_path)

reference_gene_ids = pd.Index(
    first_gene_data["gene_id"].astype(str),
    name="gene_id",
)
number_of_genes = len(reference_gene_ids)

if reference_gene_ids.duplicated().any():
    raise RuntimeError("Reference protein-coding gene IDs are not unique.")

# Preallocate arrays to avoid the large memory overhead of concatenating
# hundreds of separate pandas Series objects.
raw_counts_array = np.empty(
    (number_of_samples, number_of_genes),
    dtype=np.int64,
)
tpm_array = np.empty(
    (number_of_samples, number_of_genes),
    dtype=np.float32,
)
library_sizes = np.empty(number_of_samples, dtype=np.float64)

# Store the first sample.
raw_counts_array[0, :] = first_gene_data["unstranded"].to_numpy(dtype=np.int64)
tpm_array[0, :] = first_gene_data["tpm_unstranded"].to_numpy(dtype=np.float32)
library_sizes[0] = first_library_size

# Parse the remaining samples.
for row_number in tqdm(
    range(1, number_of_samples),
    desc="Parsing STAR-Counts files",
):
    file_path = gdc_metadata_selected.loc[row_number, "local_path"]
    gene_data, library_size = read_star_counts_file(file_path)

    current_gene_ids = pd.Index(gene_data["gene_id"].astype(str))

    if current_gene_ids.equals(reference_gene_ids):
        aligned_gene_data = gene_data
    else:
        # Defensive alignment in case a file presents genes in a different order.
        aligned_gene_data = gene_data.set_index("gene_id").reindex(reference_gene_ids)

        if aligned_gene_data[["unstranded", "tpm_unstranded"]].isna().any().any():
            missing_gene_count = aligned_gene_data["unstranded"].isna().sum()
            raise RuntimeError(
                f"{Path(file_path).name} is missing {missing_gene_count:,} "
                "reference protein-coding genes."
            )

    raw_counts_array[row_number, :] = aligned_gene_data[
        "unstranded"
    ].to_numpy(dtype=np.int64)
    tpm_array[row_number, :] = aligned_gene_data[
        "tpm_unstranded"
    ].to_numpy(dtype=np.float32)
    library_sizes[row_number] = library_size

patient_ids = pd.Index(
    gdc_metadata_selected["case_submitter_id"].astype(str),
    name="case_submitter_id",
)

X_raw_counts = pd.DataFrame(
    raw_counts_array,
    index=patient_ids,
    columns=reference_gene_ids,
)

X_tpm = pd.DataFrame(
    tpm_array,
    index=patient_ids,
    columns=reference_gene_ids,
)

# Create the gene annotation table.
gene_annotation = first_gene_data[
    ["gene_id", "gene_name", "gene_type"]
].copy()
gene_annotation["ensembl_gene_id_without_version"] = (
    gene_annotation["gene_id"].astype(str).str.split(".").str[0]
)

# Create patient/sample metadata with the same index and order as the matrices.
sample_metadata = gdc_metadata_selected.copy()
sample_metadata["library_size_all_annotated_genes"] = library_sizes
sample_metadata = sample_metadata.set_index("case_submitter_id", drop=True)
sample_metadata.index = sample_metadata.index.astype(str)
sample_metadata.index.name = "case_submitter_id"

# Essential matrix integrity checks.
if not X_raw_counts.index.equals(X_tpm.index):
    raise RuntimeError("Raw-count and TPM sample orders do not match.")
if not X_raw_counts.columns.equals(X_tpm.columns):
    raise RuntimeError("Raw-count and TPM gene orders do not match.")
if not X_raw_counts.index.equals(sample_metadata.index):
    raise RuntimeError("Expression matrices and metadata sample orders do not match.")
if (X_raw_counts.to_numpy() < 0).any():
    raise RuntimeError("Negative raw counts were detected.")
if not np.isfinite(X_tpm.to_numpy()).all():
    raise RuntimeError("Non-finite TPM values were detected.")
if X_tpm.isna().any().any():
    raise RuntimeError("Missing TPM values were detected.")

print(f"Samples: {X_tpm.shape[0]:,}")
print(f"Protein-coding genes: {X_tpm.shape[1]:,}")
print("\nClass distribution:")
print(sample_metadata["class_label"].value_counts().sort_index())

gene_annotation.to_csv(
    OUTPUT_DIR / "tcga_kidney_gene_annotation.csv",
    index=False,
)
sample_metadata.to_csv(
    OUTPUT_DIR / "tcga_kidney_sample_metadata.csv",
)

display(gene_annotation.head())
display(sample_metadata.head())

Parsing STAR-Counts files:   0%|          | 0/888 [00:00<?, ?it/s]

Samples: 889
Protein-coding genes: 19,962

Class distribution:
class_label
KICH     66
KIRC    533
KIRP    290
Name: count, dtype: int64


,gene_id,gene_name,gene_type,ensembl_gene_id_without_version
0,ENSG00000000003.15,TSPAN6,protein_coding,ENSG00000000003
1,ENSG00000000005.6,TNMD,protein_coding,ENSG00000000005
2,ENSG00000000419.13,DPM1,protein_coding,ENSG00000000419
3,ENSG00000000457.14,SCYL3,protein_coding,ENSG00000000457
4,ENSG00000000460.17,C1orf112,protein_coding,ENSG00000000460


,project_id,class_label,case_id,sample_id,sample_submitter_id,sample_type,tissue_type,file_id,file_name,file_size,md5sum,access,workflow_type,local_path,library_size_all_annotated_genes
case_submitter_id,,,,,,,,,,,,,,,
TCGA-KL-8323,TCGA-KICH,KICH,83b3060c-2449-4581-88ef-817d126e4525,8aa1a5f6-3548-494a-857b-e08124aeb9a9,TCGA-KL-8323-01A,Primary Tumor,Tumor,a4b800e3-b6ff-4823-86da-ea9755d88eac,a3bd8e23-8f88-4d92-871e-57878d42048e.rna_seq.a...,4232251,ab05a17399b6167934c7428abdd87e65,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...,59997960.0
TCGA-KL-8324,TCGA-KICH,KICH,0697436f-e487-45db-b0bc-ad9246f70196,16c2976b-c2c5-40f0-8d7d-8ecfb4a15317,TCGA-KL-8324-01A,Primary Tumor,Tumor,7b5bdde2-cbc5-4f11-9dff-3217eb69734c,6c0c2bda-2fd3-4294-8e4a-7ffad2c8f16e.rna_seq.a...,4225880,d7d528d88c38b2476d0d62c5824e3389,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...,55552279.0
TCGA-KL-8325,TCGA-KICH,KICH,6101ffe6-2ef6-4256-9d6b-a7c545836995,79c48bea-33e0-49e6-84f7-6103da654642,TCGA-KL-8325-01A,Primary Tumor,Tumor,f65e0c6a-05c4-409b-a446-85b90671c06e,9ec7c4f6-5fdd-4307-a5a9-b93a18e03ee0.rna_seq.a...,4231750,6887ae3affba94de5ef870c8ff808ae4,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...,64781564.0
TCGA-KL-8326,TCGA-KICH,KICH,03f3dd32-e1d1-485b-a968-3f55798f4d46,59703768-8be1-47a7-8c7d-d794c0936019,TCGA-KL-8326-01A,Primary Tumor,Tumor,9452f420-4244-4518-b607-3442361d065a,3dc24ff9-20bd-4c63-8152-7e43a0804943.rna_seq.a...,4216775,810b40c7ab72f8f5f313443398773e52,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...,67388622.0
TCGA-KL-8327,TCGA-KICH,KICH,02979422-5149-4750-ad5f-483e0bec6ac5,f35e66ec-7aa5-477c-abf7-07c35c90f7de,TCGA-KL-8327-01A,Primary Tumor,Tumor,0161b0d1-3fb0-45e8-bed0-df99aacaaac1,718f1665-6b2c-4d07-9dc5-93ca8e5c2bb0.rna_seq.a...,4205351,6529d9d90e70e534246f1871c69d1e09,open,STAR - Counts,/content/drive/MyDrive/TCGA_Kidney_RNASeq/gdc_...,53959542.0


# CELL 6: SAVE THE PRESERVED RAW-COUNT AND TPM SOURCE MATRICES

In [10]:
# ============================================================
# CELL 6: SAVE THE PRESERVED RAW-COUNT AND TPM SOURCE MATRICES
# ============================================================

def save_source_matrix(matrix, stem):
    """
    Save a full source matrix according to the output settings in Cell 1.

    Parameters
    ----------
    matrix : pandas.DataFrame
        Sample-by-gene matrix.
    stem : str
        Filename without extension.
    """
    if SAVE_SOURCE_MATRICES_AS_COMPRESSED_CSV:
        compressed_path = OUTPUT_DIR / f"{stem}.csv.gz"
        print(f"Saving {compressed_path.name}...")
        matrix.to_csv(
            compressed_path,
            compression="gzip",
        )

    if SAVE_SOURCE_MATRICES_AS_UNCOMPRESSED_CSV:
        uncompressed_path = OUTPUT_DIR / f"{stem}.csv"
        print(f"Saving {uncompressed_path.name}...")
        matrix.to_csv(uncompressed_path)

    if SAVE_SOURCE_MATRICES_AS_PARQUET:
        parquet_path = OUTPUT_DIR / f"{stem}.parquet"
        print(f"Saving {parquet_path.name}...")
        try:
            matrix.to_parquet(parquet_path)
        except Exception as error:
            print(
                f"Parquet save was skipped because it failed: {error!r}\n"
                "The CSV output remains available."
            )


# These files preserve both expression representations obtained from each
# GDC STAR-Counts file before downstream filtering or MAD ranking.
save_source_matrix(
    X_raw_counts,
    "tcga_kidney_raw_counts",
)
save_source_matrix(
    X_tpm,
    "tcga_kidney_tpm",
)

print("Source-matrix saving complete.")

Saving tcga_kidney_raw_counts.csv.gz...
Saving tcga_kidney_tpm.csv.gz...
Source-matrix saving complete.


# CELL 7: CALCULATE CPM FROM RAW COUNTS

In [11]:
# ============================================================
# CELL 7: CALCULATE CPM FROM RAW COUNTS
# ============================================================

# CPM is calculated from the preserved raw unstranded counts.
#
# IMPORTANT:
# The numerator matrix contains protein-coding genes, but each sample's
# denominator was calculated in Cell 5 using raw counts across ALL annotated
# Ensembl gene rows before protein-coding restriction.
#
# CPM_ij = (raw count for gene j in sample i / library size for sample i) * 1,000,000

library_size_series = sample_metadata[
    "library_size_all_annotated_genes"
].astype(np.float64)

if (library_size_series <= 0).any():
    raise RuntimeError("At least one sample has a non-positive library size.")

X_cpm = (
    X_raw_counts
    .div(library_size_series, axis="index")
    .mul(1_000_000.0)
    .astype(np.float32)
)

if not X_cpm.index.equals(X_tpm.index):
    raise RuntimeError("CPM and TPM sample orders do not match.")
if not X_cpm.columns.equals(X_tpm.columns):
    raise RuntimeError("CPM and TPM gene orders do not match.")
if not np.isfinite(X_cpm.to_numpy()).all():
    raise RuntimeError("Non-finite CPM values were detected.")

# The complete CPM matrix is optional because it can be reconstructed exactly
# from the saved raw counts and the saved library sizes.
if (
    SAVE_SOURCE_MATRICES_AS_COMPRESSED_CSV
    or SAVE_SOURCE_MATRICES_AS_UNCOMPRESSED_CSV
    or SAVE_SOURCE_MATRICES_AS_PARQUET
):
    save_source_matrix(
        X_cpm,
        "tcga_kidney_cpm",
    )

# Optionally save the complete unfiltered transformed matrices.
if SAVE_FULL_TRANSFORMED_MATRICES:
    X_log2_tpm_full = np.log2(X_tpm + 1.0).astype(np.float32)
    X_log2_cpm_full = np.log2(X_cpm + 1.0).astype(np.float32)

    save_source_matrix(
        X_log2_tpm_full,
        "tcga_kidney_log2_tpm_full_unfiltered",
    )
    save_source_matrix(
        X_log2_cpm_full,
        "tcga_kidney_log2_cpm_full_unfiltered",
    )

print("CPM calculation complete.")
print(
    f"CPM matrix dimensions: "
    f"{X_cpm.shape[0]:,} samples x {X_cpm.shape[1]:,} genes"
)

Saving tcga_kidney_cpm.csv.gz...
CPM calculation complete.
CPM matrix dimensions: 889 samples x 19,962 genes


# CELL 8: CREATE ONE STRATIFIED PATIENT-LEVEL SPLIT

In [12]:
# ============================================================
# CELL 8: CREATE ONE STRATIFIED PATIENT-LEVEL SPLIT
# ============================================================

# The split is generated ONCE from patient IDs and class labels.
# The same IDs are then used for both the TPM and CPM branches.

all_patient_ids = sample_metadata.index.to_numpy()
all_class_labels = sample_metadata["class_label"].to_numpy()

train_patient_ids, validation_patient_ids = train_test_split(
    all_patient_ids,
    test_size=VALIDATION_FRACTION,
    random_state=RANDOM_SEED,
    stratify=all_class_labels,
    shuffle=True,
)

train_patient_ids = pd.Index(
    train_patient_ids,
    name="case_submitter_id",
)
validation_patient_ids = pd.Index(
    validation_patient_ids,
    name="case_submitter_id",
)

# Confirm that the sets are disjoint and collectively exhaustive.
if set(train_patient_ids) & set(validation_patient_ids):
    raise RuntimeError("Training and validation patient IDs overlap.")

if set(train_patient_ids) | set(validation_patient_ids) != set(all_patient_ids):
    raise RuntimeError("Training and validation IDs do not cover all patients.")

# Record the split in the metadata.
sample_metadata["data_split"] = "not_assigned"
sample_metadata.loc[train_patient_ids, "data_split"] = "training"
sample_metadata.loc[validation_patient_ids, "data_split"] = "validation"

if (sample_metadata["data_split"] == "not_assigned").any():
    raise RuntimeError("At least one patient was not assigned to a split.")

split_summary = (
    sample_metadata.groupby(["data_split", "class_label"])
    .size()
    .rename("patients")
    .reset_index()
)

print("Stratified split summary:")
display(split_summary)

sample_metadata.to_csv(
    OUTPUT_DIR / "tcga_kidney_sample_metadata_with_split.csv",
)

split_summary.to_csv(
    OUTPUT_DIR / "tcga_kidney_split_summary.csv",
    index=False,
)

Stratified split summary:


,data_split,class_label,patients
0,training,KICH,46
1,training,KIRC,373
2,training,KIRP,203
3,validation,KICH,20
4,validation,KIRC,160
5,validation,KIRP,87


# CELL 9: TRAINING-ONLY FILTERING AND MAD-RANKING FUNCTION

In [13]:
# ============================================================
# CELL 9: TRAINING-ONLY FILTERING AND MAD-RANKING FUNCTION
# ============================================================

def calculate_column_mad(dataframe):
    """
    Calculate median absolute deviation independently for every gene.

    MAD_j = median_i(|x_ij - median_i(x_ij)|)

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Training expression matrix after log transformation.

    Returns
    -------
    pandas.Series
        MAD value for each gene.
    """
    values = dataframe.to_numpy(dtype=np.float32, copy=False)
    column_medians = np.median(values, axis=0)
    absolute_deviations = np.abs(values - column_medians)
    mad_values = np.median(absolute_deviations, axis=0)

    return pd.Series(
        mad_values,
        index=dataframe.columns,
        name="training_mad",
        dtype=np.float32,
    )


def prepare_expression_branch(
    linear_expression,
    branch_name,
    expression_units,
):
    """
    Prepare one complete expression branch without validation leakage.

    Processing order
    ----------------
    1. Use the already established common patient split.
    2. On TRAINING data only, retain genes with expression > 1 in at
       least 10% of training samples.
    3. Apply log2(expression + 1) to training and validation matrices.
    4. On TRAINING data only, calculate MAD for retained genes.
    5. Select the top N genes by training MAD.
    6. Apply the training-derived gene list to validation data.
    7. Save ready-to-model training and held-out validation CSV files.

    Parameters
    ----------
    linear_expression : pandas.DataFrame
        Untransformed TPM or CPM matrix.
    branch_name : str
        Filename-safe branch name.
    expression_units : str
        Human-readable expression unit, either 'TPM' or 'CPM'.

    Returns
    -------
    dict
        Selected matrices, gene statistics, and branch summary.
    """
    if not linear_expression.index.equals(sample_metadata.index):
        raise ValueError(
            f"{branch_name}: expression and metadata sample orders do not match."
        )

    # --------------------------------------------------------
    # Step A: Apply the common split.
    # --------------------------------------------------------
    X_train_linear = linear_expression.loc[train_patient_ids].copy()
    X_validation_linear = linear_expression.loc[validation_patient_ids].copy()

    # --------------------------------------------------------
    # Step B: Learn the low-expression filter from TRAINING only.
    # --------------------------------------------------------
    training_proportion_above_threshold = (
        X_train_linear.gt(EXPRESSION_THRESHOLD).mean(axis=0)
    )

    genes_passing_expression_filter = training_proportion_above_threshold.index[
        training_proportion_above_threshold
        >= MIN_TRAINING_PROPORTION_ABOVE_THRESHOLD
    ]

    if len(genes_passing_expression_filter) == 0:
        raise RuntimeError(
            f"{branch_name}: no genes passed the training expression filter."
        )

    X_train_filtered_linear = X_train_linear.loc[
        :,
        genes_passing_expression_filter,
    ]
    X_validation_filtered_linear = X_validation_linear.loc[
        :,
        genes_passing_expression_filter,
    ]

    # --------------------------------------------------------
    # Step C: Apply the branch transformation.
    #
    # This transformation itself uses no learned population parameter:
    #   primary branch     = log2(TPM + 1)
    #   sensitivity branch = log2(CPM + 1)
    # --------------------------------------------------------
    X_train_log = np.log2(
        X_train_filtered_linear.astype(np.float32) + 1.0
    ).astype(np.float32)

    X_validation_log = np.log2(
        X_validation_filtered_linear.astype(np.float32) + 1.0
    ).astype(np.float32)

    # --------------------------------------------------------
    # Step D: Rank genes by MAD using TRAINING data only.
    # --------------------------------------------------------
    training_mad = calculate_column_mad(X_train_log)

    number_to_select = min(N_TOP_MAD_GENES, len(training_mad))

    # Use a stable secondary sort by gene ID so ties are reproducible.
    mad_ranking = (
        training_mad.rename_axis("gene_id")
        .reset_index()
        .sort_values(
            by=["training_mad", "gene_id"],
            ascending=[False, True],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    mad_ranking["mad_rank"] = np.arange(1, len(mad_ranking) + 1)

    selected_gene_ids = pd.Index(
        mad_ranking.loc[: number_to_select - 1, "gene_id"],
        name="gene_id",
    )

    # --------------------------------------------------------
    # Step E: Apply the training-derived gene list to BOTH sets.
    # --------------------------------------------------------
    X_train_selected = X_train_log.loc[:, selected_gene_ids].copy()
    X_validation_selected = X_validation_log.loc[:, selected_gene_ids].copy()

    if not X_train_selected.columns.equals(X_validation_selected.columns):
        raise RuntimeError(
            f"{branch_name}: training and validation gene orders do not match."
        )

    if X_train_selected.isna().any().any():
        raise RuntimeError(f"{branch_name}: missing values in training matrix.")
    if X_validation_selected.isna().any().any():
        raise RuntimeError(f"{branch_name}: missing values in validation matrix.")

    # --------------------------------------------------------
    # Step F: Build complete gene-level audit information.
    # --------------------------------------------------------
    gene_statistics = pd.DataFrame(
        {
            "gene_id": linear_expression.columns.astype(str),
            "training_proportion_above_1": (
                training_proportion_above_threshold
                .reindex(linear_expression.columns)
                .to_numpy()
            ),
        }
    )

    gene_statistics["passed_training_expression_filter"] = (
        gene_statistics["training_proportion_above_1"]
        >= MIN_TRAINING_PROPORTION_ABOVE_THRESHOLD
    )

    gene_statistics = gene_statistics.merge(
        mad_ranking,
        on="gene_id",
        how="left",
        validate="one_to_one",
    )

    gene_statistics["selected_for_nsga_ii_input"] = (
        gene_statistics["gene_id"].isin(selected_gene_ids)
    )

    gene_statistics = gene_statistics.merge(
        gene_annotation[
            [
                "gene_id",
                "gene_name",
                "gene_type",
                "ensembl_gene_id_without_version",
            ]
        ],
        on="gene_id",
        how="left",
        validate="one_to_one",
    )

    selected_gene_table = (
        gene_statistics.loc[
            gene_statistics["selected_for_nsga_ii_input"]
        ]
        .sort_values("mad_rank")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Step G: Add labels and metadata for convenient ML input.
    #
    # The final CSV files contain metadata columns followed by genes.
    # The outcome column is named 'class_label'.
    # --------------------------------------------------------
    export_metadata_columns = [
        "project_id",
        "class_label",
        "sample_id",
        "sample_submitter_id",
        "sample_type",
        "library_size_all_annotated_genes",
        "data_split",
    ]

    training_export = sample_metadata.loc[
        train_patient_ids,
        export_metadata_columns,
    ].join(X_train_selected, how="left")

    validation_export = sample_metadata.loc[
        validation_patient_ids,
        export_metadata_columns,
    ].join(X_validation_selected, how="left")

    # Ordinary CSV is used here because the selected matrices are much smaller
    # and are ready to load directly into the NSGA-II/classifier notebook.
    training_file = OUTPUT_DIR / f"{branch_name}_training.csv"
    validation_file = OUTPUT_DIR / f"{branch_name}_heldout_validation.csv"
    selected_genes_file = OUTPUT_DIR / f"{branch_name}_selected_genes.csv"
    all_gene_statistics_file = OUTPUT_DIR / f"{branch_name}_gene_statistics.csv"

    training_export.to_csv(training_file)
    validation_export.to_csv(validation_file)
    selected_gene_table.to_csv(selected_genes_file, index=False)
    gene_statistics.to_csv(all_gene_statistics_file, index=False)

    branch_summary = {
        "branch_name": branch_name,
        "expression_units_before_log": expression_units,
        "transformation": f"log2({expression_units} + 1)",
        "training_patients": len(X_train_selected),
        "validation_patients": len(X_validation_selected),
        "starting_protein_coding_genes": linear_expression.shape[1],
        "genes_passing_training_expression_filter": len(
            genes_passing_expression_filter
        ),
        "genes_selected_by_training_mad": len(selected_gene_ids),
        "expression_threshold": EXPRESSION_THRESHOLD,
        "minimum_training_proportion_above_threshold": (
            MIN_TRAINING_PROPORTION_ABOVE_THRESHOLD
        ),
        "random_seed": RANDOM_SEED,
        "validation_fraction": VALIDATION_FRACTION,
        "training_file": str(training_file),
        "validation_file": str(validation_file),
        "selected_genes_file": str(selected_genes_file),
    }

    print(f"\nCompleted branch: {branch_name}")
    print(
        f"  Starting protein-coding genes: "
        f"{linear_expression.shape[1]:,}"
    )
    print(
        f"  Passed training-only expression filter: "
        f"{len(genes_passing_expression_filter):,}"
    )
    print(
        f"  Retained after training-only MAD ranking: "
        f"{len(selected_gene_ids):,}"
    )
    print(
        f"  Training matrix: {X_train_selected.shape[0]:,} x "
        f"{X_train_selected.shape[1]:,}"
    )
    print(
        f"  Validation matrix: {X_validation_selected.shape[0]:,} x "
        f"{X_validation_selected.shape[1]:,}"
    )

    return {
        "summary": branch_summary,
        "training_expression": X_train_selected,
        "validation_expression": X_validation_selected,
        "selected_gene_ids": selected_gene_ids,
        "selected_gene_table": selected_gene_table,
        "all_gene_statistics": gene_statistics,
        "training_export": training_export,
        "validation_export": validation_export,
    }

# CELL 10: PRIMARY PIPELINE — log2(TPM + 1)

In [14]:
# ============================================================
# CELL 10: PRIMARY PIPELINE — log2(TPM + 1)
# ============================================================

# This is the prespecified primary expression representation.
primary_tpm_results = prepare_expression_branch(
    linear_expression=X_tpm,
    branch_name="primary_log2_tpm",
    expression_units="TPM",
)


Completed branch: primary_log2_tpm
  Starting protein-coding genes: 19,962
  Passed training-only expression filter: 14,983
  Retained after training-only MAD ranking: 3,000
  Training matrix: 622 x 3,000
  Validation matrix: 267 x 3,000


# CELL 11: SENSITIVITY PIPELINE — log2(CPM + 1)

In [15]:
# ============================================================
# CELL 11: SENSITIVITY PIPELINE — log2(CPM + 1)
# ============================================================

# This sensitivity branch starts from the preserved raw unstranded counts,
# normalizes each sample to CPM, and then applies log2(CPM + 1).
sensitivity_cpm_results = prepare_expression_branch(
    linear_expression=X_cpm,
    branch_name="sensitivity_log2_cpm",
    expression_units="CPM",
)


Completed branch: sensitivity_log2_cpm
  Starting protein-coding genes: 19,962
  Passed training-only expression filter: 15,132
  Retained after training-only MAD ranking: 3,000
  Training matrix: 622 x 3,000
  Validation matrix: 267 x 3,000


# CELL 12: VERIFY IDENTICAL PATIENT SPLITS AND SUMMARIZE BRANCHES

In [16]:
# ============================================================
# CELL 12: VERIFY IDENTICAL PATIENT SPLITS AND SUMMARIZE BRANCHES
# ============================================================

# Both branches must contain exactly the same training patients and exactly
# the same held-out validation patients.
if not primary_tpm_results["training_expression"].index.equals(
    sensitivity_cpm_results["training_expression"].index
):
    raise RuntimeError("TPM and CPM training patient orders differ.")

if not primary_tpm_results["validation_expression"].index.equals(
    sensitivity_cpm_results["validation_expression"].index
):
    raise RuntimeError("TPM and CPM validation patient orders differ.")

primary_genes = set(primary_tpm_results["selected_gene_ids"])
sensitivity_genes = set(sensitivity_cpm_results["selected_gene_ids"])

selected_gene_intersection = primary_genes & sensitivity_genes
selected_gene_union = primary_genes | sensitivity_genes

selected_gene_jaccard = (
    len(selected_gene_intersection) / len(selected_gene_union)
    if selected_gene_union
    else np.nan
)

branch_summary = pd.DataFrame(
    [
        primary_tpm_results["summary"],
        sensitivity_cpm_results["summary"],
    ]
)

branch_summary["selected_gene_overlap_count"] = len(
    selected_gene_intersection
)
branch_summary["selected_gene_jaccard"] = selected_gene_jaccard

branch_summary.to_csv(
    OUTPUT_DIR / "tpm_cpm_preprocessing_branch_summary.csv",
    index=False,
)

# Save the genes selected in both branches.
overlap_table = gene_annotation.loc[
    gene_annotation["gene_id"].isin(selected_gene_intersection)
].copy()

overlap_table.to_csv(
    OUTPUT_DIR / "genes_selected_in_both_tpm_and_cpm_branches.csv",
    index=False,
)

print("Branch summary:")
display(branch_summary)

print(
    f"Genes selected in both branches: "
    f"{len(selected_gene_intersection):,}"
)
print(f"Selected-gene Jaccard similarity: {selected_gene_jaccard:.4f}")

print("\nFinal output files:")
for output_file in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {output_file.name}")

Branch summary:


,branch_name,expression_units_before_log,transformation,training_patients,validation_patients,starting_protein_coding_genes,genes_passing_training_expression_filter,genes_selected_by_training_mad,expression_threshold,minimum_training_proportion_above_threshold,random_seed,validation_fraction,training_file,validation_file,selected_genes_file,selected_gene_overlap_count,selected_gene_jaccard
0,primary_log2_tpm,TPM,log2(TPM + 1),622,267,19962,14983,3000,1.0,0.1,42,0.3,/content/drive/MyDrive/TCGA_Kidney_RNASeq/proc...,/content/drive/MyDrive/TCGA_Kidney_RNASeq/proc...,/content/drive/MyDrive/TCGA_Kidney_RNASeq/proc...,2524,0.726122
1,sensitivity_log2_cpm,CPM,log2(CPM + 1),622,267,19962,15132,3000,1.0,0.1,42,0.3,/content/drive/MyDrive/TCGA_Kidney_RNASeq/proc...,/content/drive/MyDrive/TCGA_Kidney_RNASeq/proc...,/content/drive/MyDrive/TCGA_Kidney_RNASeq/proc...,2524,0.726122


Genes selected in both branches: 2,524
Selected-gene Jaccard similarity: 0.7261

Final output files:
  gdc_star_counts_all_eligible_files.csv
  gdc_star_counts_downloaded_one_file_per_patient.csv
  gdc_star_counts_one_file_per_patient.csv
  genes_selected_in_both_tpm_and_cpm_branches.csv
  primary_log2_tpm_gene_statistics.csv
  primary_log2_tpm_heldout_validation.csv
  primary_log2_tpm_selected_genes.csv
  primary_log2_tpm_training.csv
  sensitivity_log2_cpm_gene_statistics.csv
  sensitivity_log2_cpm_heldout_validation.csv
  sensitivity_log2_cpm_selected_genes.csv
  sensitivity_log2_cpm_training.csv
  tcga_kidney_cpm.csv.gz
  tcga_kidney_gene_annotation.csv
  tcga_kidney_raw_counts.csv.gz
  tcga_kidney_sample_metadata.csv
  tcga_kidney_sample_metadata_with_split.csv
  tcga_kidney_split_summary.csv
  tcga_kidney_tpm.csv.gz
  tpm_cpm_preprocessing_branch_summary.csv


# CELL 13: OPTIONAL STANDARDIZATION FOR SCALE-SENSITIVE MODELS

In [ ]:
# ============================================================
# CELL 13: OPTIONAL STANDARDIZATION FOR SCALE-SENSITIVE MODELS
# ============================================================

# Do NOT use this step for tree-based classifiers such as:
#   - Decision trees
#   - Random forests
#   - Gradient-boosted trees
#
# Use training-derived standardization for scale-sensitive models such as:
#   - Support vector machines
#   - Logistic regression
#   - k-nearest neighbors
#   - Many neural-network architectures

from sklearn.preprocessing import StandardScaler


def standardize_training_and_validation(X_train, X_validation):
    """
    Standardize features without validation leakage.

    The scaler is fitted ONLY on the training matrix and then applied to both
    training and validation matrices.

    Returns
    -------
    X_train_scaled : pandas.DataFrame
    X_validation_scaled : pandas.DataFrame
    fitted_scaler : sklearn.preprocessing.StandardScaler
    """
    if not X_train.columns.equals(X_validation.columns):
        raise ValueError("Training and validation gene columns do not match.")

    scaler = StandardScaler()

    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        index=X_train.index,
        columns=X_train.columns,
    )

    X_validation_scaled = pd.DataFrame(
        scaler.transform(X_validation),
        index=X_validation.index,
        columns=X_validation.columns,
    )

    return X_train_scaled, X_validation_scaled, scaler


# EXAMPLE FOR AN SVM USING THE PRIMARY TPM BRANCH:
#
# X_train_tpm_scaled, X_validation_tpm_scaled, tpm_scaler = (
#     standardize_training_and_validation(
#         primary_tpm_results["training_expression"],
#         primary_tpm_results["validation_expression"],
#     )
# )
#
# For the planned gradient-boosted-tree NSGA-II fitness model, leave the
# expression matrices unstandardized.

# CELL 14: EXAMPLE — RELOAD A READY-TO-MODEL OUTPUT FILE

In [ ]:
# ============================================================
# CELL 14: EXAMPLE — RELOAD A READY-TO-MODEL OUTPUT FILE
# ============================================================

# This example shows how to separate predictors and labels when beginning
# the NSGA-II/classifier stage in a later notebook.

primary_training_data = pd.read_csv(
    OUTPUT_DIR / "primary_log2_tpm_training.csv",
    index_col="case_submitter_id",
)

primary_validation_data = pd.read_csv(
    OUTPUT_DIR / "primary_log2_tpm_heldout_validation.csv",
    index_col="case_submitter_id",
)

NON_GENE_COLUMNS = [
    "project_id",
    "class_label",
    "sample_id",
    "sample_submitter_id",
    "sample_type",
    "library_size_all_annotated_genes",
    "data_split",
]

X_train_primary = primary_training_data.drop(
    columns=NON_GENE_COLUMNS
)
y_train_primary = primary_training_data["class_label"].copy()

X_validation_primary = primary_validation_data.drop(
    columns=NON_GENE_COLUMNS
)
y_validation_primary = primary_validation_data["class_label"].copy()

if not X_train_primary.columns.equals(X_validation_primary.columns):
    raise RuntimeError("Reloaded training and validation genes do not match.")

print(f"Primary training predictors: {X_train_primary.shape}")
print(f"Primary validation predictors: {X_validation_primary.shape}")
print("\nTraining labels:")
print(y_train_primary.value_counts().sort_index())
print("\nValidation labels:")
print(y_validation_primary.value_counts().sort_index())

# NEXT:
# NSGA-II + Decision Tree + Survival Pipeline